# LLMOps: Prompt Management, Evaluation Pipelines, Cost & Telemetry

Operationalizing Large Language Model applications in production introduces distinct operational challenges:
1. **Prompt Management & Versioning**: Treating prompts as version-controlled software assets with environments (`dev`, `staging`, `prod`).
2. **Cost, Token & Latency Telemetry**: Real-time tracking of token counts and operational expenditures across model providers.
3. **Automated LLM Evaluation Pipelines**: Running regression test suites before promoting prompt templates to production.
4. **Guardrail Operations & Tracing**: Distributed tracing of multi-turn LLM chains (Langfuse / OpenTelemetry patterns).

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('07-mlops/llmops/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import llmops_tools
print('llmops_tools loaded')

## 1. Prompt Version Registry & Promotion
Decoupling prompt templates from application code and promoting versions across environments.

In [ ]:
from llmops_tools import PromptRegistry

registry = PromptRegistry()

# Version 1 of customer support prompt
p1 = registry.register_prompt(
    name="support_triage",
    template="Categorize this customer ticket as 'Billing' or 'Tech': {ticket_text}",
    input_variables=["ticket_text"],
    model_name="gpt-4o-mini",
    temperature=0.0
)
print(f"Registered {p1.name} v{p1.version} in env: {p1.environment}")

# Promote v1 to production
registry.promote("support_triage", version=1, environment="prod")
print(f"Promoted v1 to: {p1.environment}")

# Version 2 with reasoning guidelines
p2 = registry.register_prompt(
    name="support_triage",
    template="You are an expert triage agent. Output exactly one category: Billing, Tech, or General.\nTicket: {ticket_text}",
    input_variables=["ticket_text"],
    model_name="gpt-4o-mini",
    temperature=0.0
)
print(f"Registered {p2.name} v{p2.version} in env: {p2.environment}")

# Promote v2 to production (demoting v1 to archived)
registry.promote("support_triage", version=2, environment="prod")
print(f"Active production prompt is now v{registry.get_active_prompt('support_triage', 'prod').version}")

## 2. Token, Cost & Latency Telemetry
Tracking operational costs and P95 latency across production calls.

In [ ]:
from llmops_tools import LLMTelemetryTracer

tracer = LLMTelemetryTracer()

# Record simulated production traffic
tracer.record_call("support_triage", prompt_version=2, model_name="gpt-4o-mini", prompt_tokens=250, completion_tokens=15, latency_ms=120.0)
tracer.record_call("support_triage", prompt_version=2, model_name="gpt-4o-mini", prompt_tokens=310, completion_tokens=20, latency_ms=145.0)
tracer.record_call("document_summary", prompt_version=1, model_name="gpt-4o", prompt_tokens=4500, completion_tokens=600, latency_ms=1850.0)

summary = tracer.get_summary()
print("--- LLMOps Operational Telemetry ---")
print(f"Total API Calls  : {summary['total_calls']}")
print(f"Total Tokens Used: {summary['total_tokens']:,}")
print(f"Total Cost       : ${summary['total_cost_usd']:.4f}")
print(f"Mean Latency     : {summary['mean_latency_ms']:.1f} ms")
print(f"P95 Latency      : {summary['p95_latency_ms']:.1f} ms")

## 3. Automated LLM Regression Evaluation Pipeline
Running unit assertions on prompt candidates before deploying to production.

In [ ]:
from llmops_tools import EvalTestCase, run_prompt_eval_pipeline

def mock_llm_inference(rendered_prompt):
    # Simulated model response
    if "credit card charged twice" in rendered_prompt:
        return "Billing"
    if "error 500 on checkout" in rendered_prompt:
        return "Tech"
    return "General"

test_suite = [
    EvalTestCase(inputs={"ticket_text": "My credit card charged twice for one order"}, expected_substring="Billing"),
    EvalTestCase(inputs={"ticket_text": "I see an internal error 500 on checkout page"}, expected_substring="Tech"),
    EvalTestCase(inputs={"ticket_text": "Where is your headquarters located?"}, expected_substring="General")
]

eval_report = run_prompt_eval_pipeline(p2, mock_llm_inference, test_suite)
print("--- Evaluation Pipeline Report ---")
for k, v in eval_report.items():
    print(f"  {k:20s}: {v}")